# 1. Libraries & Sample Data
The first step is to load our Python Libraries and download the sample data. The dataset represents Apple stock price (1d bars) for the year 2010

In [ ]:
# Load Python Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from IPython.display import display, HTML

# for dataframe display
pd.set_option("display.max_rows", None)


def display_df(df):
    # Puts the scrollbar next to the DataFrame
    display(
        HTML(
            "<div style='height: 200px; overflow: auto; width: fit-content'>"
            + df.to_html()
            + "</div>"
        )
    )

In [ ]:
# Download Sample Data AAPL_2009-2010_6m_RAW_1d.csv
apple_filename = "./../AAPL_2009-2010_6m_RAW_1d.csv"
data = pd.read_csv(apple_filename)
# Use pandas's to_datetime() to convert any columns that are in a datetime format
data["Date"] = pd.to_datetime(data["Date"])

In [ ]:
data.info()

# 2. Exploratory Data Analysis
Next, we want to analyze our data. Display the data as a dataframe, and plot some relevant data so you can get an idea of what our dataset looks like.

In [ ]:
# Display as Dataframe
display_df(data)

In [ ]:
# Index data by Date
data.set_index("Date", inplace=True)
display_df(data)

In [ ]:
# Plot the Close Data
data["Close"].plot()

# 3. Data Cleaning
Next, we need to clean our data for training our model. This requires removal of NaN values.

In [ ]:
# Check for null values
print("Number of Null Values =\n", data.isnull().sum())

In [ ]:
# forward fill missing values
data = data.ffill()
display_df(data)

In [ ]:
# Check for null values to ensure it is now zero
print("Number of Null Values after fix =\n", data.isnull().sum())

In [ ]:
data.reset_index().to_csv("./../e2_data_CLEAN.csv")

# 4. Feature Definition
Now that we have cleaned our stock data, we can define some financial indicators, or "features" to train our model on. We will be calculating some popular indicators: 20-day Close Moving Average, 5-day Close Moving Average, 20-day Close Bollinger Bands, and 20-day Historical Volatility of Log Returns of Close Price. 

In [ ]:
data["MA5"] = data["Adj Close"].rolling(window=5).mean()
data["MA20"] = data["Adj Close"].rolling(window=20).mean()
data["STD20"] = data["Adj Close"].rolling(window=20).std()
data["BB_upper"] = data["MA20"] + 2 * data["STD20"]
data["BB_lower"] = data["MA20"] - 2 * data["STD20"]
data["Log_Ret"] = np.log(data["Adj Close"] / data["Adj Close"].shift(1))
number_trading_days_annually = 252
data["Vol20"] = data["Log_Ret"].rolling(window=20).std() * np.sqrt(
    number_trading_days_annually
)
display_df(data)

In [ ]:
data.head(40)

In [ ]:
# Remove rows with MA=NaN
data = data.dropna()

In [ ]:
data.head()

In [ ]:
# Plot Features on One Chart: Close, MA20, BB Upper, BB Lower, Vol20
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])

# add Apple's Closing price
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["Adj Close"],
        mode="lines",
        name="Apple's closing price",
    ),
    secondary_y=False,
)

# add Apple's MA20 price
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["MA20"],
        mode="lines",
        name="Apple's MA20 price",
    ),
    secondary_y=False,
)

# add Apple's BB Upper
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["BB_upper"],
        mode="lines",
        name="Apple's BB_upper price",
    ),
    secondary_y=False,
)

# add Apple's BB Lower
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["BB_lower"],
        mode="lines",
        name="Apple's BB_lower price",
    ),
    secondary_y=False,
)

# add Apple's Volatility on secondary y-axis
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["Vol20"],
        mode="lines",
        name="Apple's Vol20 (annualized)",
        line=dict(dash="dot"),
    ),
    secondary_y=True,
)

# customize layout
fig.update_layout(
    title="AAPL Closing price vs MA20 vs BB upper vs BB Lower vs Vol20",
    xaxis_title="Day",
    template="plotly_dark",
)
fig.update_yaxes(title_text="Price ($)", secondary_y=False)
fig.update_yaxes(title_text="Volatility (annualized)", secondary_y=True)

fig.show()